# 2 — Construcción de Silver

Silver tipa, deduplica y valida. Los registros aceptados se integran con `MERGE`; los rechazados quedan en cuarentena con una causa explícita. La corrección de la transacción 42 permite observar un `UPDATE` real.

In [0]:
dbutils.widgets.text("student_id", "alumno01")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"])
dbutils.widgets.text("expected_batch_id", "batch_002")
dbutils.widgets.text("job_run_id", "interactive")

In [0]:
%run ../common/config

In [0]:
%run ../common/quality_rules

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
bronze_all = qualified_table(config, "bronze_transactions_all")
assert spark.catalog.tableExists(bronze_all), "Falta bronze_transactions_all. Ejecutá la tarea 01."
customers = qualified_table(config, "silver_customers")
products = qualified_table(config, "silver_products")
transactions = qualified_table(config, "silver_transactions")
quarantine = qualified_table(config, "silver_transactions_quarantine")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {customers} USING DELTA AS
SELECT customer_id, country, segment, created_date, email
FROM (
  SELECT try_cast(customer_id AS BIGINT) customer_id, country, segment,
         try_cast(created_date AS DATE) created_date, lower(trim(email)) email,
         row_number() OVER (PARTITION BY customer_id ORDER BY _ingested_at DESC) rn
  FROM {qualified_table(config, 'bronze_customers')}
) WHERE rn = 1 AND customer_id IS NOT NULL
""")
spark.sql(f"""
CREATE OR REPLACE TABLE {products} USING DELTA AS
SELECT product_id, category, price
FROM (
  SELECT try_cast(product_id AS BIGINT) product_id, category, try_cast(price AS DECIMAL(12,2)) price,
         row_number() OVER (PARTITION BY product_id ORDER BY _ingested_at DESC) rn
  FROM {qualified_table(config, 'bronze_products')}
) WHERE rn = 1 AND product_id IS NOT NULL AND price > 0
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
raw_columns = ["transaction_id", "customer_id", "product_id", "event_ts", "amount", "payment_channel", "device_id", "is_fraud", "source_batch_id", "updated_at"]
typed = add_transaction_types(spark.table(bronze_all).dropDuplicates(raw_columns))
known_customers = spark.table(customers).select(F.col("customer_id").alias("known_customer"))
known_products = spark.table(products).select(F.col("product_id").alias("known_product"))
checked = (typed
    .join(known_customers, F.col("customer_id_typed") == F.col("known_customer"), "left")
    .join(known_products, F.col("product_id_typed") == F.col("known_product"), "left"))
checked = add_quality_reason(checked)
record_key = F.coalesce(F.col("transaction_id_typed").cast("string"), F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c), F.lit("<NULL>")) for c in raw_columns]), 256))
latest = (checked.withColumn("_record_key", record_key)
    .withColumn("_rn", F.row_number().over(Window.partitionBy("_record_key").orderBy(F.col("updated_at_typed").desc_nulls_last(), F.col("source_batch_id").desc())))
    .where("_rn = 1"))

In [0]:
valid = (latest.where(F.col("quality_reason").isNull()).select(
    F.col("transaction_id_typed").alias("transaction_id"),
    F.col("customer_id_typed").alias("customer_id"),
    F.col("product_id_typed").alias("product_id"),
    F.col("event_ts_typed").alias("event_ts"),
    F.col("amount_typed").alias("amount"), "payment_channel", "device_id",
    F.col("is_fraud_typed").alias("is_fraud"), "source_batch_id",
    F.col("updated_at_typed").alias("updated_at"), F.current_timestamp().alias("processed_at")))
spark.sql(f"""CREATE TABLE IF NOT EXISTS {transactions} (
 transaction_id BIGINT, customer_id BIGINT, product_id BIGINT, event_ts TIMESTAMP, amount DECIMAL(12,2),
 payment_channel STRING, device_id STRING, is_fraud INT, source_batch_id STRING, updated_at TIMESTAMP, processed_at TIMESTAMP
) USING DELTA""")
valid.createOrReplaceTempView("_valid_transactions")
spark.sql(f"""MERGE INTO {transactions} t USING _valid_transactions s ON t.transaction_id = s.transaction_id
WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
rejected = latest.where(F.col("quality_reason").isNotNull()).select(
    *raw_columns, "quality_reason", F.current_timestamp().alias("rejected_at"))
spark.sql(f"""CREATE TABLE IF NOT EXISTS {quarantine} (
 transaction_id STRING, customer_id STRING, product_id STRING, event_ts STRING, amount STRING,
 payment_channel STRING, device_id STRING, is_fraud STRING, source_batch_id STRING, updated_at STRING,
 quality_reason STRING, rejected_at TIMESTAMP
) USING DELTA""")
rejected.createOrReplaceTempView("_rejected_transactions")
spark.sql(f"""MERGE INTO {quarantine} t USING _rejected_transactions s
ON t.transaction_id <=> s.transaction_id AND t.source_batch_id <=> s.source_batch_id AND t.quality_reason = s.quality_reason
WHEN NOT MATCHED THEN INSERT *""")
display(spark.table(quarantine).groupBy("source_batch_id", "quality_reason").count().orderBy("source_batch_id", "quality_reason"))
display(spark.table(transactions).groupBy("source_batch_id").count().orderBy("source_batch_id"))
display(spark.sql(f"DESCRIBE HISTORY {transactions}"))

source_batch_id,quality_reason,count
batch_002,INVALID_AMOUNT,1
batch_002,UNKNOWN_CUSTOMER,1
initial,INVALID_AMOUNT,51


source_batch_id,count
batch_002,201
initial,49948


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-10-03T12:50:38.000Z,72546623639108,oskierkomm@gmail.com,MERGE,"Map(predicate -> [""(transaction_id#17793L = transaction_id#17771L)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(updated_at#17777 > updated_at#17802)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","List(820155019096133, bigdata_myrna_silver_gold, 1100180262900947, 286445586657808, 72546623639108, manual, 7474648255332020)",List(4486634479199606),6b08dea0-e0d1-466b-90b4-a66425f8d312,1003-124536-1c519dvm-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 7024, materializeSourceTimeMs -> 1528, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3519, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 50149, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1882)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
1,2026-10-03T12:47:16.000Z,72546623639108,oskierkomm@gmail.com,MERGE,"Map(predicate -> [""(transaction_id#11758L = transaction_id#11736L)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(updated_at#11742 > updated_at#11767)"",""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","List(820155019096133, bigdata_myrna_silver_gold, 581853062379729, 777026971723499, 72546623639108, manual, 7474648255332020)",List(4486634479199606),8d712182-7fad-4aee-9eb2-cb8f448b35c7,1003-124536-1c519dvm-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 887975, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 7879, materializeSourceTimeMs -> 2969, numTargetRowsInserted -> 50149, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3260, numTargetRowsUpdated -> 0, numOutputRows -> 50149, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 50149, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1511)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
0,2026-10-03T12:47:06.000Z,72546623639108,oskierkomm@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""io.unitycatalog.tableId"":""d3d9a31f-3620-4db7-836c-38d6b7f6a278"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-93f37476-b913-41c8-97e0-c1d69219df6c"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-e2b9bd7e-0d36-4d16-a9ae-734797554d58""}, statsOnLoad -> false)","List(820155019096133, bigdata_myrna_silver_gold, 581853062379729, 777026971723499, 72546623639108, manual, 7474648255332020)",List(4486634479199606),77ef2fa0-5895-4815-9c40-b54d708cbc74,1003-124536-1c519dvm-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
